# API paper intent v1

目标：验证 `src/api` 能把 dashboard/model 输出转换成 paper/sim target-position intent，并通过 PaperBrokerAdapter 做模拟执行。

边界：v1 不连接真实 broker，不下真实订单；ledger 只记录 intent 与 paper execution result。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.api import (
    PaperBrokerAdapter,
    build_target_position_intent_from_dashboard,
    execute_paper_intent,
    write_intent_ledger,
)

CARD = Path('research/api_paper_intent_v1')
CARD.mkdir(parents=True, exist_ok=True)
dashboard_path = Path('research/schedule_model_dashboard_refresh_v1/dashboard_export.json')
dashboard_payload = json.loads(dashboard_path.read_text(encoding='utf-8'))
dashboard_payload['top_models'][0]

{'rank': 1,
 'model': 'random_forest_clf__all_features',
 'base_model': 'random_forest_clf',
 'feature_set': 'all_features',
 'return_pct': 49.032787769965715,
 'benchmark_return_pct': 62.1472468532841,
 'relative_return_pct': -13.114459083318389,
 'max_drawdown_pct': 21.677243377022705,
 'sharpe': 3.8534654423822032,
 'avg_position': 0.7407635422717518,
 'turnover': 3.1283151044281827,
 'stability_score': 54.214257763652995}

In [2]:
intent = build_target_position_intent_from_dashboard(
    dashboard_payload,
    instrument='SOXL',
    min_position=0.0,
    max_position=0.75,
)
intent.to_dict()

{'instrument': 'SOXL',
 'target_position': 0.7407635422717518,
 'source_model': 'random_forest_clf__all_features',
 'source_artifact': 'matrix_card',
 'mode': 'paper',
 'schema_version': 'api.execution_intent.v1',
 'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
 'metadata': {'raw_avg_position': 0.7407635422717518,
  'min_position': 0.0,
  'max_position': 0.75,
  'base_model': 'random_forest_clf',
  'feature_set': 'all_features',
  'stability_score': 54.214257763652995,
  'max_drawdown_pct': 21.677243377022705,
  'dashboard_schema_version': 'dashboard.model_matrix.v1'}}

In [3]:
broker = PaperBrokerAdapter(cash=10000.0, shares=0)
result = execute_paper_intent(intent, broker=broker, price=100.0)
(CARD / 'execution_intent.json').write_text(json.dumps(intent.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'paper_execution_result.json').write_text(json.dumps(result.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
write_intent_ledger(CARD / 'intent_ledger.json', intent=intent, result=result)
result.to_dict()

{'schema_version': 'api.paper_execution_result.v1',
 'status': 'paper_executed',
 'broker_name': 'paper',
 'intent': {'instrument': 'SOXL',
  'target_position': 0.7407635422717518,
  'source_model': 'random_forest_clf__all_features',
  'source_artifact': 'matrix_card',
  'mode': 'paper',
  'schema_version': 'api.execution_intent.v1',
  'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
  'metadata': {'raw_avg_position': 0.7407635422717518,
   'min_position': 0.0,
   'max_position': 0.75,
   'base_model': 'random_forest_clf',
   'feature_set': 'all_features',
   'stability_score': 54.214257763652995,
   'max_drawdown_pct': 21.677243377022705,
   'dashboard_schema_version': 'dashboard.model_matrix.v1'}},
 'snapshot': {'cash': 2600.0, 'shares': 74, 'price': 100.0}}

In [4]:
summary = {
    'schema_version': 'api_paper_intent_review.v1',
    'source_dashboard': str(dashboard_path),
    'instrument': intent.instrument,
    'mode': intent.mode,
    'source_model': intent.source_model,
    'target_position': intent.target_position,
    'paper_shares': result.snapshot.shares,
    'paper_position': result.snapshot.position,
    'live_broker_enabled': False,
    'guardrail': intent.guardrail,
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'api_paper_intent_review.v1',
 'source_dashboard': 'research/schedule_model_dashboard_refresh_v1/dashboard_export.json',
 'instrument': 'SOXL',
 'mode': 'paper',
 'source_model': 'random_forest_clf__all_features',
 'target_position': 0.7407635422717518,
 'paper_shares': 74,
 'paper_position': 0.74,
 'live_broker_enabled': False,
 'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。'}

## 中文结论

- 做了什么：读取 dashboard export 的 top model，生成 SOXL paper target-position intent，并通过 PaperBrokerAdapter 模拟执行。
- 核心指标：source model=`random_forest_clf__all_features`，target_position=0.7408，paper shares=74，paper position=0.7400。
- API 边界：mode=`paper`；live_broker_enabled=`False`；guardrail=`v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。`。
- 输出：`execution_intent.json`、`paper_execution_result.json`、`intent_ledger.json`、`analysis_summary.json`。
- 是否晋升：否。本轮只验证 paper/sim API contract，不允许真实 broker 下单。
- 下一步：补 production notebook，把 schedule -> dashboard -> paper intent 作为可回放生产候选流程固定下来。